# 异质性分析-周期
将AED因子数据分割为大市值组和小市值组，
分别进行组合构建，分桶，FM回归  

## 导入库

In [45]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
import polars as pl
import numpy as np
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
dotenv.load_dotenv()

True

## 超参数

In [46]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'baseline1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "connectorx"

# 异质性组
T_GROUP = 1 # 0表示牛市，1表示熊市


## 读取数据
读取MA因子数据和指数回报数据

In [47]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552


In [48]:
index_rtr = pl.read_database_uri(
    '''
    SELECT indexcd, month, idxrtn 
    FROM trade_data.index_monthly 
    WHERE indexcd = '000300'
    ''',
    uri=CONNECTION_URL,
    engine=ENGINE
)

index_rtr.head()

indexcd,month,idxrtn
str,date,f64
"""000300""",2005-01-01,0.0
"""000300""",2005-02-01,0.089132
"""000300""",2005-03-01,-0.094026
"""000300""",2005-04-01,-0.010404
"""000300""",2005-05-01,-0.081993


## 处理数据

### 市值数据重命名

## 划分数据
按照date，生成所有portfolio的中位数，然后根据中位数，划分大市值组和小市值组

In [49]:
# ===================== 2. 读取并预处理数据（Polars核心） =====================
# 读取CSV文件（替换为你的数据路径）
# 筛选沪深300数据（假设指数代码为000300，可根据实际调整）
df_hs300 = index_rtr.filter(pl.col("indexcd") == "000300").sort("month")

# Polars实现1%/99%分位数缩尾（清洗极端值）
def winsorize_series(series: pl.Expr) -> pl.Expr:
    q_low = series.quantile(0.01)
    q_high = series.quantile(0.99)
    return series.clip(q_low, q_high)

# 添加清洗后的回报率列
df_hs300 = df_hs300.with_columns(
    pl.col("idxrtn").pipe(winsorize_series).alias("idxrtn_adj")
)

# 转换为Statsmodels适配的格式（提取时间序列）
# 注意：Statsmodels需要Pandas Series（带DatetimeIndex）
ts_returns = df_hs300.select("month", "idxrtn_adj").to_pandas().set_index("month")["idxrtn_adj"]

# ===================== 3. 拟合马尔可夫切换模型 =====================
# 核心模型设定（适配沪深300特征）
model = MarkovRegression(
    endog=ts_returns,          # 清洗后的沪深300收益率序列
    k_regimes=2,               # 2个状态：牛市/熊市
    switching_variance=True,   # 不同状态波动率不同（关键）
    trend="n"                  # 无趋势项（月度收益率无需趋势）
)

# 极大似然估计拟合模型（添加disp=False减少冗余输出）
result = model.fit(disp=False)

/home/frank/miniconda3/envs/thesis_env/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [50]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                        Markov Switching Model Results                        
==============================================================================
Dep. Variable:             idxrtn_adj   No. Observations:                  240
Model:               MarkovRegression   Log Likelihood                 293.878
Date:                Wed, 18 Mar 2026   AIC                           -579.756
Time:                        21:26:38   BIC                           -565.834
Sample:                    01-01-2005   HQIC                          -574.147
                         - 12-01-2024                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0025      0.001      4.286      0.000       0.001       0.004
                             Regime 1 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0129      0.002      5.235      0.000       0.008       0.018
                         Regime transition parameters                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
p[0->0]        0.9724      0.023     42.439      0.000       0.927       1.017
p[1->0]        0.0539      0.035      1.558      0.119      -0.014       0.122
==============================================================================

Warnings:
[1] Covariance matrix calculated using numerical (complex-step) differentiation.
"""

In [51]:
# ===================== 4. 识别牛熊市状态（Polars整合结果） =====================
# 提取状态概率并转换为Polars DataFrame
regime_probs = result.smoothed_marginal_probabilities
prob_df = pl.DataFrame({
    "month": regime_probs.index,
    "regime0_prob": regime_probs.iloc[:, 0].values,
    "regime1_prob": regime_probs.iloc[:, 1].values
})

# 将month列转换为date类型以匹配df_hs300的数据类型
prob_df = prob_df.with_columns(
    pl.col("month").cast(pl.Date).alias("month")
)

# 将状态概率合并回原数据
df_hs300 = df_hs300.join(prob_df, on="month", how="left")

# 判定牛市状态（均值更高的状态为牛市）
# 通过实际数据计算各状态的平均收益率来确定牛熊市
regime_0_data = df_hs300.filter(pl.col("regime0_prob") >= 0.5)
regime_1_data = df_hs300.filter(pl.col("regime1_prob") >= 0.5)

regime_0_mean = regime_0_data["idxrtn_adj"].mean()
regime_1_mean = regime_1_data["idxrtn_adj"].mean()

# 收益率更高的状态为牛市
bull_regime_idx = 0 if regime_0_mean > regime_1_mean else 1

# 添加牛熊市标签：1=牛市（概率≥0.5），0=熊市
df_hs300 = df_hs300.with_columns(
    pl.when(
        pl.col(f"regime{bull_regime_idx}_prob") >= 0.5
    ).then(1).otherwise(0).alias("bull_bear_label")
)

In [52]:
df_hs300 = df_hs300.select(pl.col('month').alias('date'),pl.col('bull_bear_label'))

和ma_df合并，保存为 MA因子.parquet 文件

In [53]:
df_hs300.head()

date,bull_bear_label
date,i32
2005-01-01,0
2005-02-01,0
2005-03-01,0
2005-04-01,0
2005-05-01,0


In [54]:
ma_df = ma_df.join(df_hs300,on=['date'],how='left')
ma_df = ma_df.filter(pl.col('bull_bear_label') == T_GROUP).drop('bull_bear_label')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2006-11-01,"""002031""",0.649186,0.0552
2015-10-01,"""002170""",0.597621,0.16923
2007-01-01,"""600288""",0.796684,0.179123
2006-12-01,"""600113""",0.759379,0.107097
2007-04-01,"""600008""",0.709344,0.134805


## 保存数据

In [55]:
ma_df.write_parquet(SAVE_BASE_DIR + '/MA因子.parquet')

**保存后，运行2，3，4 notebook，查看异质性结果** 